## 1. Install Libraries

In [1]:
!pip install -q scikit-learn
!pip install -q joblib

## 2. Imports

In [2]:
import os
import gc
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.cluster import MiniBatchKMeans

warnings.filterwarnings("ignore")

## 3. Project Paths

In [21]:
PROJECT_PATH = "/content/drive/MyDrive/FinSight_AI"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data"
)

PROCESSED_PATH = os.path.join(
    DATA_PATH,
    "processed"
)

EMBEDDING_PATH = os.path.join(
    DATA_PATH,
    "embeddings"
)

EMBEDDING_BATCH_PATH = os.path.join(
    EMBEDDING_PATH,
    "embedding_batches"
)

CLUSTER_PATH = os.path.join(
    DATA_PATH,
    "clusters"
)

CLUSTER_BATCH_PATH = os.path.join(
    CLUSTER_PATH,
    "cluster_batches"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

REPORT_PATH = os.path.join(
    PROJECT_PATH,
    "reports"
)

## 4. Create Output Folders

In [4]:
os.makedirs(
    CLUSTER_PATH,
    exist_ok=True
)

os.makedirs(
    CLUSTER_BATCH_PATH,
    exist_ok=True
)

print("Folders Ready")

Folders Ready


## 5. Load Embedding Manifest

In [5]:
manifest_df = pd.read_parquet(

    os.path.join(

        EMBEDDING_PATH,

        "embedding_manifest.parquet"

    )

)

manifest_df = manifest_df[
    manifest_df["embedding_file"] != "test_embedding.npy"
].reset_index(drop=True)

manifest_df.head()

,batch_number,start_row,end_row,rows,embedding_dimension,embedding_file,metadata_file
0,1,0,4999,5000,768,embedding_batch_0001.npy,metadata_batch_0001.parquet
1,2,5000,9999,5000,768,embedding_batch_0002.npy,metadata_batch_0002.parquet
2,3,10000,14999,5000,768,embedding_batch_0003.npy,metadata_batch_0003.parquet
3,4,15000,19999,5000,768,embedding_batch_0004.npy,metadata_batch_0004.parquet
4,5,20000,24999,5000,768,embedding_batch_0005.npy,metadata_batch_0005.parquet


## 6. Configuration

In [6]:
EMBEDDING_DIM = 768

TOTAL_BATCHES = len(manifest_df)

N_CLUSTERS = 3000

MODEL_BATCH_SIZE = 5000

RANDOM_STATE = 42

print(f"Embedding Dimension : {EMBEDDING_DIM}")

print(f"Total Batches : {TOTAL_BATCHES}")

print(f"Clusters : {N_CLUSTERS}")

Embedding Dimension : 768
Total Batches : 644
Clusters : 3000


## 7. Initialize MiniBatchKMeans

In [7]:
kmeans = MiniBatchKMeans(

    n_clusters=N_CLUSTERS,

    batch_size=MODEL_BATCH_SIZE,

    random_state=RANDOM_STATE,

    init="k-means++",

    n_init="auto",

    reassignment_ratio=0.01,

    max_no_improvement=20,

    verbose=0

)

print(kmeans)

MiniBatchKMeans(batch_size=5000, max_no_improvement=20, n_clusters=3000,
                random_state=42)


## 8. Checkpoint Function

In [8]:
CHECKPOINT_PATH = os.path.join(

    CLUSTER_PATH,

    "kmeans_training_progress.json"

)

def save_checkpoint(

    completed_batches

):

    checkpoint = {

        "completed_batches": completed_batches,

        "last_updated": time.strftime(

            "%Y-%m-%d %H:%M:%S"

        )

    }

    with open(

        CHECKPOINT_PATH,

        "w"

    ) as f:

        json.dump(

            checkpoint,

            f,

            indent=4

        )

## 9. Training Loop (Incremental Learning)

In [9]:
overall_start = time.time()

for batch_number in tqdm(

    range(TOTAL_BATCHES),

    desc="Training MiniBatchKMeans"

):

    embedding_file = os.path.join(

        EMBEDDING_BATCH_PATH,

        manifest_df.loc[

            batch_number,

            "embedding_file"

        ]

    )

    embeddings = np.load(

        embedding_file,

        mmap_mode="r"

    )

    embeddings = np.asarray(

        embeddings,

        dtype=np.float32

    )

    kmeans.partial_fit(

        embeddings

    )

    save_checkpoint(

        batch_number + 1

    )

    elapsed = time.time() - overall_start

    avg_batch = elapsed / (batch_number + 1)

    eta = (

        TOTAL_BATCHES

        - batch_number

        - 1

    ) * avg_batch / 60

    if (

        (batch_number + 1) % 25 == 0

        or

        batch_number == TOTAL_BATCHES - 1

    ):

        print()

        print("=" * 70)

        print(

            f"Batch : {batch_number+1}/{TOTAL_BATCHES}"

        )

        print(

            f"ETA : {eta:.2f} minutes"

        )

        print(

            f"Inertia : {kmeans.inertia_:,.2f}"

        )

    del embeddings

    gc.collect()

Training MiniBatchKMeans:   0%|          | 0/644 [00:00<?, ?it/s]


Batch : 25/644
ETA : 81.70 minutes
Inertia : 1,053.63

Batch : 50/644
ETA : 45.98 minutes
Inertia : 2,184.96

Batch : 75/644
ETA : 33.74 minutes
Inertia : 2,132.71

Batch : 100/644
ETA : 28.12 minutes
Inertia : 2,048.69

Batch : 125/644
ETA : 23.88 minutes
Inertia : 2,198.37

Batch : 150/644
ETA : 20.93 minutes
Inertia : 2,112.83

Batch : 175/644
ETA : 18.73 minutes
Inertia : 1,874.81

Batch : 200/644
ETA : 16.89 minutes
Inertia : 1,945.48

Batch : 225/644
ETA : 15.24 minutes
Inertia : 2,012.06

Batch : 250/644
ETA : 13.78 minutes
Inertia : 2,493.70

Batch : 275/644
ETA : 12.49 minutes
Inertia : 2,204.78

Batch : 300/644
ETA : 11.32 minutes
Inertia : 2,788.43

Batch : 325/644
ETA : 10.22 minutes
Inertia : 2,294.91

Batch : 350/644
ETA : 9.20 minutes
Inertia : 2,629.23

Batch : 375/644
ETA : 8.24 minutes
Inertia : 2,733.48

Batch : 400/644
ETA : 7.34 minutes
Inertia : 2,575.23

Batch : 425/644
ETA : 6.50 minutes
Inertia : 2,595.99

Batch : 450/644
ETA : 5.69 minutes
Inertia : 2,690.97


## 10. Save Trained Model

In [10]:
MODEL_FILE = os.path.join(

    MODEL_PATH,

    "kmeans_model.pkl"

)

joblib.dump(

    kmeans,

    MODEL_FILE

)

print()

print("="*60)

print("MiniBatchKMeans Training Completed")

print(f"Model Saved : {MODEL_FILE}")


MiniBatchKMeans Training Completed
Model Saved : /content/drive/MyDrive/FinSight_AI/models/kmeans_model.pkl


## 11. Prediction Checkpoint

In [11]:
PREDICTION_CHECKPOINT = os.path.join(

    CLUSTER_PATH,

    "cluster_prediction_progress.json"

)

def save_prediction_checkpoint(

    completed_batches,

    processed_rows

):

    checkpoint = {

        "completed_batches": completed_batches,

        "processed_rows": processed_rows,

        "last_updated": time.strftime(

            "%Y-%m-%d %H:%M:%S"

        )

    }

    with open(

        PREDICTION_CHECKPOINT,

        "w"

    ) as f:

        json.dump(

            checkpoint,

            f,

            indent=4

        )

## 12. Process One Batch

In [12]:
def process_batch(

    batch_number,

    embedding_file,

    metadata_file

):

    embeddings = np.load(

        embedding_file,

        mmap_mode="r"

    )

    embeddings = np.asarray(

        embeddings,

        dtype=np.float32

    )

    metadata = pd.read_parquet(

        metadata_file

    )

    clusters = kmeans.predict(

        embeddings

    )

    distances = kmeans.transform(

        embeddings

    )

    min_distance = np.min(

        distances,

        axis=1

    )

    confidence = 1 / (

        1 +

        min_distance

    )

    batch_df = pd.DataFrame({

        "news_id": metadata["news_id"].values,

        "cluster": clusters,

        "confidence": confidence,

        "distance_to_centroid": min_distance,

        "batch_number": batch_number,

        "row_in_batch": np.arange(

            len(metadata)

        ),

        "embedding_file": os.path.basename(

            embedding_file

        )

    })

    batch_df["is_centroid_candidate"] = False

    idx = (

        batch_df

        .groupby("cluster")["distance_to_centroid"]

        .idxmin()

    )

    batch_df.loc[

        idx,

        "is_centroid_candidate"

    ] = True

    del embeddings
    del metadata
    del distances

    gc.collect()

    return batch_df

## 13. Test One Batch

In [13]:
embedding_file = os.path.join(

    EMBEDDING_BATCH_PATH,

    manifest_df.loc[0, "embedding_file"]

)

metadata_file = os.path.join(

    EMBEDDING_BATCH_PATH,

    manifest_df.loc[0, "metadata_file"]

)

test_batch = process_batch(

    1,

    embedding_file,

    metadata_file

)

test_batch.head()

,news_id,cluster,confidence,distance_to_centroid,batch_number,row_in_batch,embedding_file,is_centroid_candidate
0,1,72,0.996382,0.003631,1,0,embedding_batch_0001.npy,True
1,2,72,0.726620,0.376235,1,1,embedding_batch_0001.npy,False
2,3,2896,0.812242,0.231161,1,2,embedding_batch_0001.npy,False
3,4,1090,0.832623,0.201024,1,3,embedding_batch_0001.npy,False
4,5,2948,0.619009,0.615485,1,4,embedding_batch_0001.npy,False


## 14. Verify Confidence

In [14]:
test_batch.describe()

,news_id,cluster,confidence,distance_to_centroid,batch_number,row_in_batch
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.0,5000.000000
mean,2500.500000,1205.893800,0.618489,0.653486,1.0,2499.500000
std,1443.520003,909.570759,0.103171,0.223464,0.0,1443.520003
min,1.000000,0.000000,0.493376,0.000002,1.0,0.000000
25%,1250.750000,368.000000,0.550070,0.554759,1.0,1249.750000
50%,2500.500000,1063.000000,0.582874,0.715637,1.0,2499.500000
75%,3750.250000,1962.000000,0.643186,0.817951,1.0,3749.250000
max,5000.000000,2999.000000,0.999998,1.026853,1.0,4999.000000


## 15. Full Production Loop

In [15]:
# import shutil

# if os.path.exists(CLUSTER_BATCH_PATH):

#     shutil.rmtree(CLUSTER_BATCH_PATH)

# os.makedirs(

#     CLUSTER_BATCH_PATH,

#     exist_ok=True

# )

# files_to_delete = [

#     "cluster_manifest.parquet",

#     "cluster_statistics.parquet",

#     "cluster_prediction_progress.json"

# ]

# for file in files_to_delete:

#     path = os.path.join(

#         CLUSTER_PATH,

#         file

#     )

#     if os.path.exists(path):

#         os.remove(path)

# report_file = os.path.join(

#     REPORT_PATH,

#     "cluster_summary.parquet"

# )

# if os.path.exists(report_file):

#     os.remove(report_file)

# rep_file = os.path.join(

#     CLUSTER_PATH,

#     "representative_headlines.parquet"

# )

# if os.path.exists(rep_file):

#     os.remove(rep_file)

# print("Old clustering results deleted.")

In [16]:
cluster_manifest = []

processed_rows = 0

overall_start = time.time()

for batch_number in tqdm(

    range(TOTAL_BATCHES),

    desc="Assigning Clusters"

):

    output_file = os.path.join(

        CLUSTER_BATCH_PATH,

        f"cluster_batch_{batch_number+1:04d}.parquet"

    )

    if os.path.exists(output_file):

        continue

    embedding_file = os.path.join(

        EMBEDDING_BATCH_PATH,

        manifest_df.loc[

            batch_number,

            "embedding_file"

        ]

    )

    metadata_file = os.path.join(

        EMBEDDING_BATCH_PATH,

        manifest_df.loc[

            batch_number,

            "metadata_file"

        ]

    )

    batch_start = time.time()

    batch_df = process_batch(

        batch_number + 1,

        embedding_file,

        metadata_file

    )

    batch_df.to_parquet(

        output_file,

        index=False

    )

    cluster_manifest.append({

        "batch_number": batch_number + 1,

        "rows": len(batch_df),

        "cluster_file": os.path.basename(

            output_file

        )

    })

    processed_rows += len(batch_df)

    save_prediction_checkpoint(

        batch_number + 1,

        processed_rows

    )

    elapsed = time.time() - overall_start

    avg_batch = elapsed / (batch_number + 1)

    eta = (

        TOTAL_BATCHES -

        batch_number -

        1

    ) * avg_batch / 60

    if (

        (batch_number + 1) % 25 == 0

        or

        batch_number == TOTAL_BATCHES - 1

    ):

        print()

        print("="*70)

        print(f"Batch : {batch_number+1}/{TOTAL_BATCHES}")

        print(f"Rows Processed : {processed_rows:,}")

        print(f"ETA : {eta:.2f} minutes")

        print(f"Average Confidence : {batch_df['confidence'].mean():.3f}")

        print(f"Batch Time : {time.time()-batch_start:.2f} sec")

    del batch_df

    gc.collect()

Assigning Clusters:   0%|          | 0/644 [00:00<?, ?it/s]


Batch : 450/644
Rows Processed : 90,000
ETA : 0.38 minutes
Average Confidence : 0.595
Batch Time : 2.54 sec

Batch : 475/644
Rows Processed : 215,000
ETA : 0.75 minutes
Average Confidence : 0.596
Batch Time : 2.48 sec

Batch : 500/644
Rows Processed : 340,000
ETA : 0.96 minutes
Average Confidence : 0.607
Batch Time : 2.46 sec

Batch : 525/644
Rows Processed : 465,000
ETA : 1.03 minutes
Average Confidence : 0.599
Batch Time : 2.44 sec

Batch : 550/644
Rows Processed : 590,000
ETA : 0.99 minutes
Average Confidence : 0.603
Batch Time : 3.56 sec

Batch : 575/644
Rows Processed : 715,000
ETA : 0.84 minutes
Average Confidence : 0.599
Batch Time : 3.55 sec

Batch : 600/644
Rows Processed : 840,000
ETA : 0.61 minutes
Average Confidence : 0.594
Batch Time : 2.57 sec

Batch : 625/644
Rows Processed : 965,000
ETA : 0.29 minutes
Average Confidence : 0.606
Batch Time : 2.86 sec

Batch : 644/644
Rows Processed : 1,055,296
ETA : 0.00 minutes
Average Confidence : 0.635
Batch Time : 0.31 sec


### Create Cluster Manifest

In [17]:
cluster_manifest = pd.DataFrame(

    cluster_manifest

)

cluster_manifest.to_parquet(

    os.path.join(

        CLUSTER_PATH,

        "cluster_manifest.parquet"

    ),

    index=False

)

cluster_manifest.head()

,batch_number,rows,cluster_file
0,433,5000,cluster_batch_0433.parquet
1,434,5000,cluster_batch_0434.parquet
2,435,5000,cluster_batch_0435.parquet
3,436,5000,cluster_batch_0436.parquet
4,437,5000,cluster_batch_0437.parquet


### Create Cluster Statistics

In [18]:
cluster_stats = []

for file in tqdm(

    sorted(

        os.listdir(

            CLUSTER_BATCH_PATH

        )

    ),

    desc="Cluster Statistics"

):

    df = pd.read_parquet(

        os.path.join(

            CLUSTER_BATCH_PATH,

            file

        )

    )

    stats = (

        df.groupby("cluster")

        .agg(

            headlines=(

                "news_id",

                "count"

            ),

            avg_confidence=(

                "confidence",

                "mean"

            ),

            min_distance=(

                "distance_to_centroid",

                "min"

            )

        )

        .reset_index()

    )

    cluster_stats.append(

        stats

    )

cluster_stats = (

    pd.concat(

        cluster_stats,

        ignore_index=True

    )

    .groupby("cluster")

    .agg(

        headlines=(

            "headlines",

            "sum"

        ),

        avg_confidence=(

            "avg_confidence",

            "mean"

        ),

        representative_distance=(

            "min_distance",

            "min"

        )

    )

    .reset_index()

)

cluster_stats.to_parquet(

    os.path.join(

        CLUSTER_PATH,

        "cluster_statistics.parquet"

    ),

    index=False

)

cluster_stats.head()

Cluster Statistics:   0%|          | 0/644 [00:00<?, ?it/s]

,cluster,headlines,avg_confidence,representative_distance
0,0,1647,0.848314,1.160776e-01
1,1,198,0.996909,8.560065e-08
2,2,790,0.998421,4.750784e-06
3,3,56,0.643269,2.107342e-08
4,4,439,0.716785,2.302391e-01


### Final Summary

In [19]:
summary = pd.DataFrame({

    "Metric":[

        "Total Headlines",

        "Total Clusters",

        "Embedding Dimension",

        "Average Cluster Size",

        "Largest Cluster",

        "Average Confidence",

        "Cluster Files"

    ],

    "Value":[

        processed_rows,

        cluster_stats.shape[0],

        EMBEDDING_DIM,

        round(

            cluster_stats["headlines"].mean(),

            2

        ),

        cluster_stats["headlines"].max(),

        round(

            cluster_stats["avg_confidence"].mean(),

            3

        ),

        TOTAL_BATCHES

    ]

})

summary.to_parquet(

    os.path.join(

        REPORT_PATH,

        "cluster_summary.parquet"

    ),

    index=False

)

summary

,Metric,Value
0,Total Headlines,1055296.000
1,Total Clusters,2913.000
2,Embedding Dimension,768.000
3,Average Cluster Size,1103.770
4,Largest Cluster,36014.000
5,Average Confidence,0.729
6,Cluster Files,644.000


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/FinSight_AI/data/processed/headline_lookup.parquet'